In [ ]:
!nvidia-smi

Fri May 22 16:05:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch, subprocess, sys

print(f"PyTorch {torch.__version__}  CUDA {torch.version.cuda}")

# ── Ultralytics ───────────────────────────────────────────────────────────────
!pip install ultralytics -q

# ── MMPose ecosystem via pip โดยตรง (ข้าม mim — Python 3.12 compat issues) ───
!pip install "mmengine>=0.10.0" -q
!pip install "mmdet>=3.1.0" "mmpose>=1.3.0" -q

# ── mmcv: ลอง wheel จาก OpenMMLab CDN หลาย combo ────────────────────────────
_cu = torch.version.cuda.replace('.', '')          # เช่น "128"
_pt = '.'.join(torch.__version__.split('.')[:2])   # เช่น "2.10"

_cdn = 'https://download.openmmlab.com/mmcv/dist'

# ลำดับ fallback: exact match → cu124/torch2.5 → cu121/torch2.4 → cu121/torch2.1
_combos = [
    (_cu, _pt),
    ('124', '2.5'),
    ('121', '2.4'),
    ('121', '2.1'),
]

_mmcv_ok = False
for _c, _p in _combos:
    _url = f'{_cdn}/cu{_c}/torch{_p}/index.html'
    print(f"mmcv: ลอง cu{_c}/torch{_p} ...")
    _r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'mmcv>=2.0.0',
         '-f', _url, '-q'],
        capture_output=True, text=True, timeout=300,
    )
    if _r.returncode == 0:
        print(f'✓ mmcv (cu{_c}/torch{_p})')
        _mmcv_ok = True
        break
    # wheel ไม่พบ — ลอง combo ถัดไป

if not _mmcv_ok:
    # PyPI fallback: มี CPU-only wheel ใน PyPI สำหรับบาง version
    print('[fallback] ลอง mmcv จาก PyPI ...')
    _r_pypi = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'mmcv>=2.0.0', '-q'],
        capture_output=True, text=True, timeout=300,
    )
    if _r_pypi.returncode == 0:
        print('✓ mmcv (PyPI)')
        _mmcv_ok = True
    else:
        print('✗ mmcv PyPI ล้มเหลว — pip stderr:')
        print(_r_pypi.stderr[-2000:] if _r_pypi.stderr else '(ไม่มี stderr)')
        print('[!] ติดตั้ง mmcv ไม่ได้ — RTMPose/HigherHRNet cell จะรันไม่ผ่าน')
        print('[!] ตรวจสอบ PyTorch/CUDA version compatibility ที่:')
        print('    https://mmcv.readthedocs.io/en/latest/get_started/installation.html')

import ultralytics; ultralytics.checks()
print("✓ Ultralytics ready" + ("  |  ✓ mmcv ready" if _mmcv_ok else "  |  ✗ mmcv NOT installed"))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import csv, json, time
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE      = '/content/drive/MyDrive/Models/Ohm-Vision'
YOLO_DATA  = f'{DRIVE}/Datasets/yolo-pose'
COCO_DATA  = f'{DRIVE}/Datasets/coco-pose'   # COCO JSON สำหรับ MMPose
OUT        = f'{DRIVE}/Pose-Training-Results'
Path(OUT).mkdir(parents=True, exist_ok=True)

# ── Fair Comparison Constants (fixed สำหรับทุกโมเดล) ─────────────────────────
MAX_EPOCHS = 100
PATIENCE   = 20      # early stop หาก metric ไม่ดีขึ้นใน N epochs
BATCH      = 32
SEED       = 42
LR0        = 1e-3

# ── Keypoint + Category metadata (2 object class, 2 keypoints ต่อ instance) ──
# body_0 / body_1 = ปลายสองด้านของ resistor หรือ wire
# ทั้ง 2 class ใช้ keypoint schema เดียวกัน (class-agnostic keypoints)
KPT_META = dict(
    dataset_name = 'ohm_vision',
    paper_info   = dict(),
    classes      = ['resistor', 'wire'],   # ← บอก MMPose ให้ load ทั้ง 2 category
    keypoint_info = {
        0: dict(name='body_0', id=0, color=[0, 255, 0], type='lower', swap='body_1'),
        1: dict(name='body_1', id=1, color=[0, 255, 0], type='upper', swap='body_0'),
    },
    skeleton_info = {0: dict(link=('body_0', 'body_1'), id=0, color=[0, 255, 0])},
    joint_weights = [1., 1.],
    sigmas        = [0.025, 0.025],
)

# ── Metrics helper (บันทึกผลทุกโมเดลลง comparison.csv) ────────────────────────
_CSV    = f'{OUT}/comparison.csv'
_FIELDS = ['model', 'pose_AP50', 'pose_AP50_95', 'box_AP50', 'box_AP50_95',
           'params_M', 'size_MB', 'stopped_epoch', 'train_min', 'inf_ms_GPU']

def save_metrics(model_name, pose_ap50, pose_ap50_95,
                 box_ap50='N/A', box_ap50_95='N/A',
                 params_m=0, size_mb=0, stopped_epoch=0, train_min=0, inf_ms=0):
    row = dict(
        model        = model_name,
        pose_AP50    = round(float(pose_ap50), 4),
        pose_AP50_95 = round(float(pose_ap50_95), 4),
        box_AP50     = box_ap50 if isinstance(box_ap50, str) else round(float(box_ap50), 4),
        box_AP50_95  = box_ap50_95 if isinstance(box_ap50_95, str) else round(float(box_ap50_95), 4),
        params_M     = round(params_m, 2),
        size_MB      = round(size_mb, 2),
        stopped_epoch= stopped_epoch,
        train_min    = round(train_min, 1),
        inf_ms_GPU   = round(inf_ms, 2),
    )
    write_header = not Path(_CSV).exists()
    with open(_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=_FIELDS)
        if write_header: w.writeheader()
        w.writerow(row)
    print(f"✓ {model_name}: pose_AP50={pose_ap50:.4f}  pose_AP50_95={pose_ap50_95:.4f}")

print(f"Ready  MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  BATCH={BATCH}  SEED={SEED}  LR0={LR0}")
print(f"Classes: {KPT_META['classes']}  |  Keypoints per instance: {len(KPT_META['keypoint_info'])}")


---
# Dataset

| Format | Path | ใช้กับ |
|---|---|---|
| YOLO pose | `yolo-pose/images/{train,val,test}/` | YOLOv8n |
| COCO JSON | `coco-pose/{train,valid,test}/` | RTMPose, HigherHRNet |

Dataset เดียวกัน แปลง format สองแบบ — split train/val/test ตรงกัน

In [ ]:
import yaml

# YOLO pose dataset YAML
data_yaml = dict(
    path      = YOLO_DATA,
    train     = 'images/train',
    val       = 'images/val',
    test      = 'images/test',
    nc        = 2,
    names     = ['resistor', 'wire'],
    kpt_shape = [2, 3],
)
with open(f'{YOLO_DATA}/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print("Dataset summary:")
print(f"{'split':8s}  {'YOLO':>6s}  {'COCO':>6s}")
for split, coco_split in [('train','train'),('val','valid'),('test','test')]:
    n_yolo = len(list(Path(f'{YOLO_DATA}/images/{split}').glob('*.*')))
    n_coco = len(list(Path(f'{COCO_DATA}/{coco_split}').glob('*.jpg')) +
                  list(Path(f'{COCO_DATA}/{coco_split}').glob('*.png')))
    print(f"  {split:8s}  {n_yolo:6d}  {n_coco:6d}")
print("✓ data.yaml written")

---
# Model 1 — YOLOv8n-pose
**Paradigm:** One-stage anchor-free (detection + pose ในขั้นตอนเดียว)  
**Input:** 640×640 full image  
**Params:** ~3.3 M  
**Metric:** Box mAP + Pose mAP (OKS)

In [ ]:
from ultralytics import YOLO
import pandas as pd

_YOLO_DIR = f'{OUT}/yolov8n_pose'

# ── Train ─────────────────────────────────────────────────────────────────────
t0 = time.time()
model = YOLO('yolov8n-pose.pt')
model.train(
    data     = f'{YOLO_DATA}/data.yaml',
    epochs   = MAX_EPOCHS,
    patience = PATIENCE,
    imgsz    = 640,
    batch    = BATCH,
    seed     = SEED,
    lr0      = LR0,
    lrf      = 0.01,
    optimizer= 'AdamW',
    cos_lr   = True,
    pose     = 18.0,
    kobj     = 2.0,
    workers  = 4,
    device   = 0,
    project  = OUT,
    name     = 'yolov8n_pose',
    exist_ok = True,
    verbose  = False,
)
train_min = (time.time() - t0) / 60
print(f"Training: {train_min:.1f} min")

# ── Stopped epoch ─────────────────────────────────────────────────────────────
stopped_ep = len(pd.read_csv(f'{_YOLO_DIR}/results.csv'))

# ── Test evaluation ───────────────────────────────────────────────────────────
best = f'{_YOLO_DIR}/weights/best.pt'
model = YOLO(best)
met = model.val(data=f'{YOLO_DATA}/data.yaml', split='test',
                imgsz=640, conf=0.5, iou=0.45, device=0, verbose=False)

# ── Inference speed (GPU) ─────────────────────────────────────────────────────
imgs = list(Path(f'{YOLO_DATA}/images/test').glob('*.jpg'))[:30]
preds = model.predict(imgs, imgsz=640, device=0, verbose=False)
inf_ms = sum(r.speed['inference'] for r in preds) / len(preds)

# ── Model stats ───────────────────────────────────────────────────────────────
params_m = sum(p.numel() for p in YOLO(best).model.parameters()) / 1e6
size_mb  = Path(best).stat().st_size / 1e6

print(f"\n{'='*50}")
print(f"YOLOv8n-pose  epoch={stopped_ep}/{MAX_EPOCHS}  train={train_min:.1f}min")
print(f"  Box  AP50={met.box.map50:.4f}  AP50:95={met.box.map:.4f}")
print(f"  Pose AP50={met.pose.map50:.4f}  AP50:95={met.pose.map:.4f}")
print(f"  {params_m:.2f}M params | {size_mb:.1f}MB | {inf_ms:.2f}ms/img (GPU)")

save_metrics('yolov8n_pose',
    pose_ap50=met.pose.map50, pose_ap50_95=met.pose.map,
    box_ap50=met.box.map50,   box_ap50_95=met.box.map,
    params_m=params_m, size_mb=size_mb,
    stopped_epoch=stopped_ep, train_min=train_min, inf_ms=inf_ms)

---
# Model 2 — RTMPose-s
**Paradigm:** Two-stage Top-Down (detect bbox → crop → pose per instance)  
**Input crop:** 256×192  
**Params:** ~5.1 M  
**Framework:** MMPose 1.x  
**Note:** Evaluation ใช้ GT bbox — แยก detection error ออกจาก pose error (top-down standard)

In [ ]:
from mmengine.runner import Runner
from mmpose.apis import init_model, inference_topdown
import torch

RTM_DIR = f'{OUT}/rtmpose_s'
Path(RTM_DIR).mkdir(parents=True, exist_ok=True)

# ── Shared pipeline components ────────────────────────────────────────────────
_simcc = dict(type='SimCCLabel', input_size=(256, 192),
              smoothing_type='gaussian', sigma=(5.66, 5.66), simcc_split_ratio=2.0)

train_pipeline = [
    dict(type='LoadImage'),
    dict(type='GetBBoxCenterScale'),
    dict(type='RandomFlip', direction='horizontal'),
    dict(type='RandomBBoxTransform', scale_factor=[0.6, 1.4], rotate_factor=80),
    dict(type='TopdownAffine', input_size=(256, 192)),
    dict(type='GenerateTarget', encoder=_simcc),
    dict(type='PackPoseInputs')]

val_pipeline = [
    dict(type='LoadImage'),
    dict(type='GetBBoxCenterScale'),
    dict(type='TopdownAffine', input_size=(256, 192)),
    dict(type='PackPoseInputs')]

# data_mode='topdown': ทำงานกับ bbox crop ต่อ instance
# metainfo=KPT_META: classes=['resistor','wire'] → load annotations ทั้ง 2 category
_ds = dict(type='CocoDataset', data_mode='topdown', metainfo=KPT_META)

# ── Full config dict ──────────────────────────────────────────────────────────
rtm_cfg = dict(
    work_dir  = RTM_DIR,
    launcher  = 'none',
    randomness= dict(seed=SEED, deterministic=False),
    env_cfg   = dict(cudnn_benchmark=False,
                     mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
                     dist_cfg=dict(backend='nccl')),
    log_level = 'INFO',
    load_from = None,
    resume    = False,

    model=dict(
        type='TopdownPoseEstimator',
        data_preprocessor=dict(
            type='PoseDataPreprocessor',
            mean=[123.675, 116.28, 103.53], std=[58.395, 57.12, 57.375],
            bgr_to_rgb=True),
        backbone=dict(
            type='CSPNeXt', arch='P5', expand_ratio=0.5,
            deepen_factor=0.167, widen_factor=0.375, out_indices=(4,),
            channel_attention=True,
            norm_cfg=dict(type='BN'), act_cfg=dict(type='SiLU', inplace=True),
            init_cfg=dict(type='Pretrained', prefix='backbone.',
                checkpoint='https://download.openmmlab.com/mmpose/v1/projects/'
                           'rtmposev1/cspnext-s_imagenet_600e.pth')),
        neck=dict(
            type='CSPNeXtPAFPN', in_channels=(128, 256), out_channels=256,
            num_csp_blocks=1, expand_ratio=0.5,
            norm_cfg=dict(type='BN'), act_cfg=dict(type='SiLU', inplace=True)),
        head=dict(
            type='RTMCCHead', in_channels=256, out_channels=2,
            input_size=(256, 192), in_featuremap_size=(8, 6),
            simcc_split_ratio=2.0, final_layer_kernel_size=7,
            gau_cfg=dict(hidden_dims=256, s=128, expansion_factor=2,
                dropout_rate=0., drop_path=0., act_fn='SiLU',
                use_rel_bias=False, pos_enc=False),
            loss=dict(type='KLDiscretLoss', use_target_weight=True,
                beta=10., label_softmax=True),
            decoder=_simcc),
        test_cfg=dict(flip_test=True)),

    train_dataloader=dict(
        batch_size=BATCH, num_workers=4, persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=True),
        dataset=dict(**_ds, data_root=COCO_DATA,
            ann_file='train/_annotations.coco.json',
            data_prefix=dict(img='train/'), pipeline=train_pipeline)),

    val_dataloader=dict(
        batch_size=BATCH, num_workers=4, persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False, round_up=False),
        dataset=dict(**_ds, data_root=COCO_DATA,
            ann_file='valid/_annotations.coco.json',
            data_prefix=dict(img='valid/'), pipeline=val_pipeline)),

    test_dataloader=dict(
        batch_size=BATCH, num_workers=4,
        sampler=dict(type='DefaultSampler', shuffle=False, round_up=False),
        dataset=dict(**_ds, data_root=COCO_DATA,
            ann_file='test/_annotations.coco.json',
            data_prefix=dict(img='test/'), pipeline=val_pipeline)),

    # classwise=True → รายงาน AP แยกต่อ class (resistor / wire) ด้วย
    val_evaluator =dict(type='CocoMetric', classwise=True,
        ann_file=f'{COCO_DATA}/valid/_annotations.coco.json'),
    test_evaluator=dict(type='CocoMetric', classwise=True,
        ann_file=f'{COCO_DATA}/test/_annotations.coco.json'),

    optim_wrapper=dict(type='OptimWrapper',
        optimizer=dict(type='AdamW', lr=LR0, weight_decay=0.05)),

    param_scheduler=[
        dict(type='LinearLR', begin=0, end=500,
             start_factor=1e-5, by_epoch=False),
        dict(type='CosineAnnealingLR', begin=0, end=MAX_EPOCHS,
             T_max=MAX_EPOCHS, by_epoch=True, eta_min=1e-7)],

    train_cfg=dict(by_epoch=True, max_epochs=MAX_EPOCHS, val_interval=1),
    val_cfg=dict(),
    test_cfg=dict(),

    default_hooks=dict(
        timer=dict(type='IterTimerHook'),
        logger=dict(type='LoggerHook', interval=50),
        param_scheduler=dict(type='ParamSchedulerHook'),
        checkpoint=dict(type='CheckpointHook', interval=1,
            save_best='coco/AP', rule='greater', max_keep_ckpts=1),
        sampler_seed=dict(type='DistSamplerSeedHook'),
        visualization=dict(type='PoseVisualizationHook', enable=False)),

    custom_hooks=[
        dict(type='EMAHook', ema_type='ExpMomentumEMA',
             momentum=0.0002, update_buffers=True),
        dict(type='EarlyStoppingHook', monitor='coco/AP',
             patience=PATIENCE, rule='greater', min_delta=0.0005)],

    vis_backends=[dict(type='LocalVisBackend')],
    visualizer=dict(type='PoseLocalVisualizer',
        vis_backends=[dict(type='LocalVisBackend')], name='visualizer'),
)

# ── Train ─────────────────────────────────────────────────────────────────────
t0 = time.time()
runner = Runner.from_cfg(rtm_cfg)
runner.train()
train_min = (time.time() - t0) / 60

# ── Best checkpoint ───────────────────────────────────────────────────────────
ckpts = sorted(Path(RTM_DIR).glob('best_coco_AP_epoch_*.pth'))
if not ckpts:
    ckpts = sorted(Path(RTM_DIR).glob('epoch_*.pth'))
best_ckpt = str(ckpts[-1]) if ckpts else None

# ── Stopped epoch ─────────────────────────────────────────────────────────────
log_jsons = sorted(Path(RTM_DIR).rglob('vis_data/scalars.json'))
stopped_ep = MAX_EPOCHS
if log_jsons:
    lines = Path(log_jsons[-1]).read_text().strip().splitlines()
    stopped_ep = json.loads(lines[-1]).get('step', MAX_EPOCHS) if lines else MAX_EPOCHS

# ── Test evaluation ───────────────────────────────────────────────────────────
rtm_cfg_test = dict(**rtm_cfg, load_from=best_ckpt)
metrics = Runner.from_cfg(rtm_cfg_test).test()
pose_ap50    = metrics.get('coco/AP50', 0)
pose_ap50_95 = metrics.get('coco/AP',   0)

# per-class AP (classwise=True ทำให้มี key เหล่านี้)
for cls in KPT_META['classes']:
    print(f"  [{cls}] AP50={metrics.get(f'coco/{cls}_AP50', 'N/A'):.4f}"
          f"  AP={metrics.get(f'coco/{cls}_AP', 'N/A'):.4f}")

# ── Inference speed (GPU, GT bbox top-down) ───────────────────────────────────
model_rtm = init_model(rtm_cfg_test, best_ckpt, device='cuda:0')
test_imgs  = list(Path(f'{COCO_DATA}/test').glob('*.jpg'))[:30]
for img in test_imgs[:5]:
    inference_topdown(model_rtm, str(img))          # warm-up
t_inf = time.time()
for img in test_imgs:
    inference_topdown(model_rtm, str(img))
inf_ms = (time.time() - t_inf) / len(test_imgs) * 1000

# ── Model stats ───────────────────────────────────────────────────────────────
ckpt_data = torch.load(best_ckpt, map_location='cpu', weights_only=False)
state     = ckpt_data.get('state_dict', ckpt_data)
params_m  = sum(v.numel() for v in state.values() if hasattr(v, 'numel')) / 1e6
size_mb   = Path(best_ckpt).stat().st_size / 1e6

print(f"\n{'='*50}")
print(f"RTMPose-s  epoch={stopped_ep}/{MAX_EPOCHS}  train={train_min:.1f}min")
print(f"  Pose AP50={pose_ap50:.4f}  AP50:95={pose_ap50_95:.4f}")
print(f"  {params_m:.2f}M params | {size_mb:.1f}MB | {inf_ms:.2f}ms/img (GPU)")

save_metrics('rtmpose_s',
    pose_ap50=pose_ap50, pose_ap50_95=pose_ap50_95,
    params_m=params_m, size_mb=size_mb,
    stopped_epoch=stopped_ep, train_min=train_min, inf_ms=inf_ms)


---
# Model 3 — HigherHRNet-w32
**Paradigm:** Bottom-Up (ทำนาย keypoints ทั้งภาพพร้อมกัน แล้วจับคู่เป็น instances)  
**Input:** 512×512 full image (ไม่ crop รายตัว — ทุก instance ในภาพเดียวกัน)  
**Params:** ~28.6 M  
**Framework:** MMPose 1.x  
**Note:** ไม่ต้องการ detector — ทำนาย heatmap + Associative Embedding สำหรับ group keypoints ทุก instance พร้อมกัน

In [ ]:
from mmengine.runner import Runner
from mmpose.apis import init_model, inference_bottomup
import torch

HRN_DIR = f'{OUT}/higherhrnet_w32'
Path(HRN_DIR).mkdir(parents=True, exist_ok=True)

# ── AssociativeEmbedding codec (bottom-up grouping) ───────────────────────────
_ae_codec = dict(
    type='AssociativeEmbedding',
    input_size=(512, 512),
    heatmap_size=(128, 128),
    use_udp=True,
    num_keypoints=2)

train_pipeline_bu = [
    dict(type='LoadImage'),
    dict(type='BottomupRandomAffine',
         input_size=(512, 512),
         shift_factor=0.16, rotate_factor=45,
         scale_factor=(0.75, 1.5), scale_type='short'),
    dict(type='RandomFlip', direction='horizontal'),
    dict(type='BottomupGetHeatmapMask'),
    dict(type='PhotometricDistortion'),
    dict(type='GenerateTarget', encoder=_ae_codec),
    dict(type='PackPoseInputs')]

val_pipeline_bu = [
    dict(type='LoadImage'),
    dict(type='BottomupResize',
         input_size=(512, 512), size_factor=32, resize_mode='expand'),
    dict(type='PackPoseInputs')]

# data_mode='bottomup': ทำนาย keypoints ทั้งภาพ — ไม่ต้องการ detector
# metainfo=KPT_META: classes=['resistor','wire'] → load annotations ทั้ง 2 category
_ds_bu = dict(type='CocoDataset', data_mode='bottomup', metainfo=KPT_META)

hrn_cfg = dict(
    work_dir  = HRN_DIR,
    launcher  = 'none',
    randomness= dict(seed=SEED, deterministic=False),
    env_cfg   = dict(cudnn_benchmark=False,
                     mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
                     dist_cfg=dict(backend='nccl')),
    log_level = 'INFO',
    load_from = None,
    resume    = False,

    model=dict(
        type='BottomupPoseEstimator',
        data_preprocessor=dict(
            type='PoseDataPreprocessor',
            mean=[123.675, 116.28, 103.53],
            std=[58.395, 57.12, 57.375],
            bgr_to_rgb=True),
        backbone=dict(
            type='HRNet',
            in_channels=3,
            extra=dict(
                stage1=dict(
                    num_modules=1, num_branches=1,
                    block='BOTTLENECK', num_blocks=(4,), num_channels=(64,)),
                stage2=dict(
                    num_modules=1, num_branches=2,
                    block='BASIC', num_blocks=(4, 4), num_channels=(32, 64)),
                stage3=dict(
                    num_modules=4, num_branches=3,
                    block='BASIC', num_blocks=(4, 4, 4), num_channels=(32, 64, 128)),
                stage4=dict(
                    num_modules=3, num_branches=4,
                    block='BASIC', num_blocks=(4, 4, 4, 4),
                    num_channels=(32, 64, 128, 256),
                    multiscale_output=True)),
            init_cfg=dict(
                type='Pretrained',
                checkpoint='https://download.openmmlab.com/mmpose/pretrain_models/hrnet_w32-36af842e.pth')),
        neck=dict(type='FeatureMapProcessor', concat=True),
        head=dict(
            type='BottomupHeatmapHead',
            in_channels=480,          # 32+64+128+256 (HRNet-w32 concat)
            num_keypoints=2,
            tag_dim=1,
            with_ae_loss=True,
            loss=dict(
                type='MultiLossFactory',
                num_joints=2,
                num_stages=1,
                ae_loss_type='exp',
                with_ae_loss=[True],
                push_loss_factor=[0.001],
                pull_loss_factor=[0.001],
                with_heatmaps_loss=[True],
                heatmaps_loss_factor=[1.0]),
            decoder=_ae_codec),
        test_cfg=dict(
            multiscale_test=False,
            flip_test=True,
            nms_dist_thr=0.05,
            shift_keypoint=False,
            output_heatmaps=False)),

    train_dataloader=dict(
        batch_size=BATCH, num_workers=4, persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=True),
        dataset=dict(**_ds_bu, data_root=COCO_DATA,
            ann_file='train/_annotations.coco.json',
            data_prefix=dict(img='train/'),
            pipeline=train_pipeline_bu)),

    val_dataloader=dict(
        batch_size=1, num_workers=4, persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False, round_up=False),
        dataset=dict(**_ds_bu, data_root=COCO_DATA,
            ann_file='valid/_annotations.coco.json',
            data_prefix=dict(img='valid/'),
            pipeline=val_pipeline_bu)),

    test_dataloader=dict(
        batch_size=1, num_workers=4,
        sampler=dict(type='DefaultSampler', shuffle=False, round_up=False),
        dataset=dict(**_ds_bu, data_root=COCO_DATA,
            ann_file='test/_annotations.coco.json',
            data_prefix=dict(img='test/'),
            pipeline=val_pipeline_bu)),

    # classwise=True → รายงาน AP แยกต่อ class (resistor / wire) ด้วย
    val_evaluator =dict(type='CocoMetric', classwise=True,
        ann_file=f'{COCO_DATA}/valid/_annotations.coco.json'),
    test_evaluator=dict(type='CocoMetric', classwise=True,
        ann_file=f'{COCO_DATA}/test/_annotations.coco.json'),

    optim_wrapper=dict(type='OptimWrapper',
        optimizer=dict(type='Adam', lr=LR0, weight_decay=0.0)),

    param_scheduler=[
        dict(type='LinearLR', begin=0, end=500,
             start_factor=1e-5, by_epoch=False),
        dict(type='MultiStepLR', begin=0, end=MAX_EPOCHS,
             milestones=[int(MAX_EPOCHS * 0.9)], gamma=0.1, by_epoch=True)],

    train_cfg=dict(by_epoch=True, max_epochs=MAX_EPOCHS, val_interval=1),
    val_cfg=dict(),
    test_cfg=dict(),

    default_hooks=dict(
        timer       =dict(type='IterTimerHook'),
        logger      =dict(type='LoggerHook', interval=50),
        param_scheduler=dict(type='ParamSchedulerHook'),
        checkpoint  =dict(type='CheckpointHook', interval=1,
                          save_best='coco/AP', rule='greater', max_keep_ckpts=1),
        sampler_seed=dict(type='DistSamplerSeedHook'),
        visualization=dict(type='PoseVisualizationHook', enable=False)),

    custom_hooks=[
        dict(type='EarlyStoppingHook', monitor='coco/AP',
             patience=PATIENCE, rule='greater', min_delta=0.0005)],

    vis_backends=[dict(type='LocalVisBackend')],
    visualizer=dict(type='PoseLocalVisualizer',
        vis_backends=[dict(type='LocalVisBackend')], name='visualizer'),
)

# ── Train ─────────────────────────────────────────────────────────────────────
t0 = time.time()
runner = Runner.from_cfg(hrn_cfg)
runner.train()
train_min = (time.time() - t0) / 60

# ── Best checkpoint ───────────────────────────────────────────────────────────
ckpts = sorted(Path(HRN_DIR).glob('best_coco_AP_epoch_*.pth'))
if not ckpts:
    ckpts = sorted(Path(HRN_DIR).glob('epoch_*.pth'))
best_ckpt = str(ckpts[-1]) if ckpts else None

# ── Stopped epoch ─────────────────────────────────────────────────────────────
log_jsons  = sorted(Path(HRN_DIR).rglob('vis_data/scalars.json'))
stopped_ep = MAX_EPOCHS
if log_jsons:
    lines      = Path(log_jsons[-1]).read_text().strip().splitlines()
    stopped_ep = json.loads(lines[-1]).get('step', MAX_EPOCHS) if lines else MAX_EPOCHS

# ── Test evaluation ───────────────────────────────────────────────────────────
hrn_cfg_test = dict(**hrn_cfg, load_from=best_ckpt)
metrics      = Runner.from_cfg(hrn_cfg_test).test()
pose_ap50    = metrics.get('coco/AP50', 0)
pose_ap50_95 = metrics.get('coco/AP',   0)

# per-class AP (classwise=True)
for cls in KPT_META['classes']:
    print(f"  [{cls}] AP50={metrics.get(f'coco/{cls}_AP50', 'N/A'):.4f}"
          f"  AP={metrics.get(f'coco/{cls}_AP', 'N/A'):.4f}")

# ── Inference speed (GPU, full-image bottom-up) ───────────────────────────────
model_hrn = init_model(hrn_cfg_test, best_ckpt, device='cuda:0')
test_imgs  = list(Path(f'{COCO_DATA}/test').glob('*.jpg'))[:30]
for img in test_imgs[:5]:
    inference_bottomup(model_hrn, str(img))         # warm-up
t_inf  = time.time()
for img in test_imgs:
    inference_bottomup(model_hrn, str(img))
inf_ms = (time.time() - t_inf) / len(test_imgs) * 1000

# ── Model stats ───────────────────────────────────────────────────────────────
ckpt_data = torch.load(best_ckpt, map_location='cpu', weights_only=False)
state     = ckpt_data.get('state_dict', ckpt_data)
params_m  = sum(v.numel() for v in state.values() if hasattr(v, 'numel')) / 1e6
size_mb   = Path(best_ckpt).stat().st_size / 1e6

print(f"\n{'='*50}")
print(f"HigherHRNet-w32  epoch={stopped_ep}/{MAX_EPOCHS}  train={train_min:.1f}min")
print(f"  Pose AP50={pose_ap50:.4f}  AP50:95={pose_ap50_95:.4f}")
print(f"  {params_m:.2f}M params | {size_mb:.1f}MB | {inf_ms:.2f}ms/img (GPU)")

save_metrics('higherhrnet_w32',
    pose_ap50=pose_ap50, pose_ap50_95=pose_ap50_95,
    params_m=params_m, size_mb=size_mb,
    stopped_epoch=stopped_ep, train_min=train_min, inf_ms=inf_ms)


---
# Pose Model Comparison

เปรียบเทียบ 3 paradigm บน dataset เดียวกัน — split/seed/epoch/patience เหมือนกันทุกโมเดล

| | YOLOv8n-pose | RTMPose-s | HigherHRNet-w32 |
|---|---|---|---|
| Paradigm | One-Stage | Top-Down | Bottom-Up |
| Input size | 640×640 full | 256×192 crop | 512×512 full |
| Detection required | ✗ (รวมอยู่แล้ว) | ✓ (GT bbox ใช้ตอน eval) | ✗ (ทำนายทั้งภาพ) |
| Framework | Ultralytics | MMPose 1.x | MMPose 1.x |
| Pretrained | ImageNet | ImageNet (CSPNeXt) | ImageNet (HRNet-w32) |

In [ ]:
import pandas as pd

df = pd.read_csv(_CSV)

# ── pretty-print ──────────────────────────────────────────────────────────────
cols_show = ['model', 'pose_AP50', 'pose_AP50_95',
             'box_AP50', 'box_AP50_95',
             'params_M', 'size_MB',
             'stopped_epoch', 'train_min', 'inf_ms_GPU']
df = df[cols_show].copy()
df = df.rename(columns={
    'pose_AP50':    'Pose AP50',
    'pose_AP50_95': 'Pose AP50:95',
    'box_AP50':     'Box AP50',
    'box_AP50_95':  'Box AP50:95',
    'params_M':     'Params (M)',
    'size_MB':      'Size (MB)',
    'stopped_epoch':'Stopped Ep.',
    'train_min':    'Train (min)',
    'inf_ms_GPU':   'Inf. ms/img',
})

print(df.to_string(index=False))
print(f"\n[saved] {_CSV}")